# Lesson 01 - Pengenalan kepada Ejen AI

Selamat datang ke pelajaran pertama dalam kursus **Ejen AI untuk Pemula**!

Sebuah **ejen AI** adalah program yang menggunakan model bahasa besar (LLM) sebagai enjin penalarannya dan boleh mengambil *tindakan* dalam dunia sebenar — memanggil API, membuat pertanyaan pangkalan data, atau menjalankan kod — untuk mencapai matlamat bagi pihak pengguna.

Dalam buku nota ini anda akan membina ejen pertama anda: sebuah **Ejen Pelancongan** yang mencadangkan destinasi percutian. Sepanjang perjalanan anda akan belajar bagaimana untuk:

1. Sambungkan kepada Perkhidmatan Ejen Azure AI Foundry menggunakan **Rangka Kerja Ejen Microsoft**.
2. Berikan ejen sebuah **alat** — fungsi Python biasa yang ia boleh panggil.
3. Jalankan ejen dan periksa responsnya.
4. Alirkan respons ejen token demi token.


## Penyediaan

Sebelum menjalankan notebook ini, pastikan anda telah:

1. **Projek Azure AI Foundry** dengan model perbualan yang telah diterapkan (contohnya `gpt-4o-mini`).
2. **Log masuk dengan Azure CLI** — jalankan `az login` di terminal anda.
3. **Tetapkan pemboleh ubah persekitaran yang diperlukan:**
   - `AZURE_AI_PROJECT_ENDPOINT` — titik hujung projek Azure AI Foundry anda.
   - `AZURE_AI_MODEL_DEPLOYMENT_NAME` — nama model yang telah anda terapkan.

Sel selanjutnya memasang pakej Python yang anda perlukan.


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity -q

In [ ]:
import logging
logging.getLogger("agent_framework.azure").setLevel(logging.ERROR)

import os
import asyncio
from typing import Annotated

from agent_framework import tool
from agent_framework.azure import AzureAIProjectAgentProvider
from azure.identity import AzureCliCredential

provider = AzureAIProjectAgentProvider(credential=AzureCliCredential())

## Mencipta Ejen Pertama Anda

Satu ejen memerlukan dua perkara:

- **Arahan** yang memberitahunya *siapa dia* dan *bagaimana untuk berkelakuan* (prompt sistem).
- **Alat** — fungsi Python yang dihias dengan `@tool` yang boleh dipanggil oleh ejen untuk mendapatkan maklumat atau melaksanakan tindakan.

Di bawah ini kami menentukan alat mudah yang mengembalikan senarai destinasi percutian popular. Ejen akan menggunakan alat ini apabila pengguna meminta cadangan perjalanan.


In [ ]:
@tool(approval_mode="never_require")
def get_destinations() -> list[str]:
    """Get a list of popular vacation destinations."""
    return [
        "Barcelona",
        "Paris",
        "Berlin",
        "Tokyo",
        "Sydney",
        "New York City",
        "Cairo",
        "Cape Town",
        "Rio de Janeiro",
        "Bali",
    ]

In [ ]:
agent = await provider.create_agent(
    tools=[get_destinations],
    name="TravelAgent",
    instructions=(
        "You are a helpful travel agent. Help users find their perfect vacation "
        "destination based on their preferences. Use the get_destinations tool "
        "to see available destinations."
    ),
)

response = await agent.run(
    "I'm looking for a warm beach destination. What do you recommend?"
)
print(response)

## Streaming Responses

Untuk pengalaman yang lebih interaktif, anda boleh **menstrim** tindak balas ejen tersebut. Daripada menunggu balasan penuh, ejen akan mengeluarkan potongan teks secara berperingkat apabila ia dijana. Ini amat berguna terutamanya dalam antara muka sembang di mana anda ingin memaparkan output secara masa nyata.


In [ ]:
async for chunk in agent.run(
    "Tell me about Tokyo as a travel destination", stream=True
):
    print(chunk, end="", flush=True)

## Summary

Dalam pelajaran ini anda telah belajar bagaimana untuk:

- **Mencipta pembekal** yang menyambung ke Azure AI Foundry Agent Service melalui `AzureAIProjectAgentProvider`.
- **Mentakrifkan alat** menggunakan dekorator `@tool` supaya ejen boleh memanggil fungsi Python anda.
- **Menjalankan ejen** dengan mesej pengguna dan mencetak responsnya.
- **Menstrim respons** untuk keluaran masa nyata.

Dalam pelajaran seterusnya kita akan meneroka kerangka kerja ejen dengan lebih mendalam dan belajar bagaimana untuk memberikan ejen alatan yang lebih berkuasa serta keupayaan penaakulan berbilang langkah.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Penafian**:  
Dokumen ini telah diterjemahkan menggunakan perkhidmatan terjemahan AI [Co-op Translator](https://github.com/Azure/co-op-translator). Walaupun kami berusaha untuk ketepatan, sila maklum bahawa terjemahan automatik mungkin mengandungi kesilapan atau ketidaktepatan. Dokumen asal dalam bahasa asalnya hendaklah dianggap sebagai sumber yang sah dan rasmi. Untuk maklumat penting, disarankan mendapatkan terjemahan profesional oleh manusia. Kami tidak bertanggungjawab terhadap sebarang salah faham atau tafsiran yang salah yang timbul daripada penggunaan terjemahan ini.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
